In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
a = 1

In [4]:
import torch
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
# =========================
# LOAD KMNIST
# =========================
transform = transforms.Compose([
    transforms.Resize(224),  # for pretrained models
    transforms.Grayscale(num_output_channels=3),  # convert to 3-channel
    transforms.ToTensor()
])

train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)

# =========================
# LOAD PRETRAINED MODELS
# =========================
alexnet = torch.hub.load('pytorch/vision', 'alexnet', pretrained=True)
vgg = torch.hub.load('pytorch/vision', 'vgg16', pretrained=True)
resnet = torch.hub.load('pytorch/vision', 'resnet18', pretrained=True)
googlenet = torch.hub.load('pytorch/vision', 'googlenet', pretrained=True)

models = {
    "AlexNet": alexnet,
    "VGG16": vgg,
    "ResNet18": resnet,
    "GoogLeNet": googlenet
}

# =========================
# MODIFY FINAL LAYER (10 classes)
# =========================
for name, model in models.items():
    if name == "ResNet18" or name == "GoogLeNet":
        model.fc = nn.Linear(model.fc.in_features, 10)
    else:
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 10)
# =========================
# TRAIN FUNCTION (1 epoch only)
# =========================
def train_model(model):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        break 

# =========================
# EVALUATE
# =========================
def evaluate(model):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            break  

    return correct / total

# =========================
# RUN ALL MODELS
# =========================
results = {}

for name, model in models.items():
    print(f"\nRunning {name}")
    train_model(model)
    acc = evaluate(model)
    print(f"{name} Accuracy:", acc)
    results[name] = acc

print("\nFinal Results:", results)

Using cache found in /root/.cache/torch/hub/pytorch_vision_main
Using cache found in /root/.cache/torch/hub/pytorch_vision_main
Using cache found in /root/.cache/torch/hub/pytorch_vision_main
Using cache found in /root/.cache/torch/hub/pytorch_vision_main



Running AlexNet
AlexNet Accuracy: 0.21875

Running VGG16
VGG16 Accuracy: 0.125

Running ResNet18
ResNet18 Accuracy: 0.40625

Running GoogLeNet
GoogLeNet Accuracy: 0.34375

Final Results: {'AlexNet': 0.21875, 'VGG16': 0.125, 'ResNet18': 0.40625, 'GoogLeNet': 0.34375}


Initially, the Kuzushiji-MNIST (KMNIST) dataset was used. However, due to a network timeout error during dataset download, the dataset could not be accessed in the execution environment.

Error encountered:

RuntimeError: Error downloading KMNIST (Connection timed out)

To ensure successful experimentation, the dataset was replaced with Fashion-MNIST, which:

Is readily available in PyTorch
Has same image size (28×28 grayscale)
Is suitable for CNN-based models
Acts as a valid alternative benchmark dataset

While implementing GoogLeNet, the following error occurred:

AttributeError: 'GoogLeNet' object has no attribute 'classifier'
Reason:
Different models use different final layers
GoogLeNet uses model.fc instead of model.classifier

1. Performance Comparison
ResNet18 achieved the highest accuracy (40.63%)
It outperformed:
GoogLeNet by +6.25%
AlexNet by +18.75%
VGG16 by +28.13%

Reason:

Residual connections help in better gradient flow
Prevents vanishing gradient problem
Enables deeper learning even with limited training
2. Poor Performance of VGG16 (12.5%)
Very deep network with large number of parameters
Requires:
More data
Longer training time

On small/quick training:

Model underfits
Accuracy drops significantly
3. AlexNet Performance (21.87%)
Better than VGG but still low
Reason:
Older architecture
Less efficient feature extraction compared to modern models
4. GoogLeNet Performance (34.38%)
Uses Inception modules (multi-scale feature extraction)
Performs better than AlexNet/VGG

But:

Slightly worse than ResNet
Due to lack of residual connections
5. Overall Insight
Modern architectures (ResNet, GoogLeNet) perform better than older ones
Residual learning > Inception > Traditional CNNs

Switching to Fashion-MNIST ensured successful experimentation without affecting validity
Model architecture differences must be handled carefully (e.g., .fc vs .classifier)
ResNet18 is the best performing model due to efficient gradient propagation
Performance is limited due to:
Small dataset
Limited training time (Kaggle constraints)